# NB03 — NumPy-style Docstrings and Sphinx Documentation

**Module 16 — Research Software Engineering**

---

## Motivation

A function without a docstring is a black box — you must read the implementation to know how to use it, what it accepts, and what can go wrong. NumPy-style docstrings are the de facto standard in scientific Python: they are human-readable, machine-parseable, and render into beautiful HTML documentation via Sphinx. Every public function in `compbio_utils` will have one.

## Intuition

A docstring is not a comment. Comments explain *how* the code works. Docstrings explain *what* the function does, *what* it accepts, and *what* it returns — from the caller's perspective, not the implementer's. The docstring is part of the function's public interface; the implementation is not.

## Biological Background

We document a biologically meaningful function: `melting_temperature(seq)`. The melting temperature (Tm) of a short DNA duplex is the temperature at which 50% of the double-stranded DNA is denatured to single strands. For short oligonucleotides (≤ 14 bp), the Wallace rule gives a quick estimate: Tm = 2°C × (A+T) + 4°C × (G+C). This is used to design PCR primers.

## Mathematical Explanation

Wallace rule: Tm (°C) = 2(nA + nT) + 4(nG + nC), where nA, nT, nG, nC are nucleotide counts.

For longer sequences (> 14 bp), the nearest-neighbour thermodynamic model is more accurate, but Wallace suffices for teaching the documentation pattern.

---

In [ ]:
import inspect
import textwrap

# A fully documented function: the gold standard
def melting_temperature(seq: str, method: str = 'wallace') -> float:
    """Estimate the melting temperature of a short DNA oligonucleotide.

    Uses the Wallace rule for short sequences (recommended for <= 14 bp):
    ``Tm = 2*(nA + nT) + 4*(nG + nC)`` degrees Celsius.

    Parameters
    ----------
    seq : str
        DNA sequence (5' to 3'). Case-insensitive. Must contain only
        the characters A, T, C, G.
    method : {'wallace'}, optional
        Calculation method. Currently only 'wallace' is implemented.
        Default is 'wallace'.

    Returns
    -------
    float
        Estimated melting temperature in degrees Celsius.

    Raises
    ------
    ValueError
        If ``seq`` is empty.
    ValueError
        If ``seq`` contains characters outside {A, T, C, G}.
    NotImplementedError
        If ``method`` is not 'wallace'.

    Notes
    -----
    The Wallace rule is an approximation valid for short, salt-adjusted
    conditions (50 mM monovalent ions). For sequences longer than 14 bp,
    use the nearest-neighbour thermodynamic model (SantaLucia 1998).

    References
    ----------
    .. [1] Wallace, R.B. et al. (1979) Nucleic Acids Res., 6(11), 3543-3557.

    Examples
    --------
    >>> melting_temperature('ATCG')
    12.0
    >>> melting_temperature('GCGCGCGCGC')
    40.0
    >>> melting_temperature('AAAA')
    8.0
    """
    if not seq:
        raise ValueError('seq must not be empty')
    seq = seq.upper()
    valid = set('ATCG')
    invalid = set(seq) - valid
    if invalid:
        raise ValueError(f'Invalid nucleotide(s): {", ".join(sorted(invalid))}')
    if method != 'wallace':
        raise NotImplementedError(f'Method {method!r} not implemented. Use "wallace".')
    n_at = seq.count('A') + seq.count('T')
    n_gc = seq.count('G') + seq.count('C')
    return float(2 * n_at + 4 * n_gc)


# Verify doctests pass
assert melting_temperature('ATCG') == 12.0
assert melting_temperature('GCGCGCGCGC') == 40.0
assert melting_temperature('AAAA') == 8.0
print('All doctest examples pass.')

In [ ]:
# Parse and display docstring sections programmatically
import re

def parse_numpy_docstring(fn):
    """Extract sections from a NumPy-style docstring."""
    raw = inspect.getdoc(fn) or ''
    sections = {}
    current_section = 'Summary'
    current_lines = []
    
    # NumPy section headers: a line followed by a line of dashes
    section_pattern = re.compile(r'^([A-Za-z][\w ]+)\s*$')
    dash_pattern = re.compile(r'^[-]+\s*$')
    
    lines = raw.split('\n')
    i = 0
    while i < len(lines):
        line = lines[i]
        # Check if next line is dashes (= this is a section header)
        if i + 1 < len(lines) and dash_pattern.match(lines[i + 1]):
            sections[current_section] = '\n'.join(current_lines).strip()
            current_section = line.strip()
            current_lines = []
            i += 2  # skip header and dashes
        else:
            current_lines.append(line)
            i += 1
    
    sections[current_section] = '\n'.join(current_lines).strip()
    return sections


sections = parse_numpy_docstring(melting_temperature)
print(f'Sections found: {list(sections.keys())}\n')
for name, content in sections.items():
    print(f'=== {name} ===')
    print(content[:200] + ('...' if len(content) > 200 else ''))
    print()

In [ ]:
# Docstring quality audit: check which standard sections are present
REQUIRED_SECTIONS = {'Parameters', 'Returns', 'Raises', 'Examples'}
RECOMMENDED_SECTIONS = {'Notes', 'References', 'See Also'}

def audit_docstring(fn):
    """Audit a function's docstring for NumPy-style completeness."""
    sections = parse_numpy_docstring(fn)
    found = set(sections.keys())
    missing_required = REQUIRED_SECTIONS - found
    missing_recommended = RECOMMENDED_SECTIONS - found
    
    present_required = REQUIRED_SECTIONS & found
    present_recommended = RECOMMENDED_SECTIONS & found
    
    score = len(present_required) / len(REQUIRED_SECTIONS)
    
    return {
        'function': fn.__name__,
        'present_required': sorted(present_required),
        'missing_required': sorted(missing_required),
        'present_recommended': sorted(present_recommended),
        'missing_recommended': sorted(missing_recommended),
        'score': score,
    }


# Test on melting_temperature (should be complete) vs a bad example
def bad_function(x):
    "Does something with x."
    return x * 2


for fn in [melting_temperature, bad_function]:
    audit = audit_docstring(fn)
    print(f"--- {audit['function']} ---")
    print(f"  Required present:  {audit['present_required']}")
    print(f"  Required missing:  {audit['missing_required']}")
    print(f"  Score: {audit['score']:.0%}")
    print()

In [ ]:
# Show the Sphinx conf.py that would be used to build docs
sphinx_conf = textwrap.dedent("""
    # docs/conf.py
    project = 'compbio_utils'
    author = 'Vinoth'
    release = '0.1.0'
    extensions = [
        'sphinx.ext.autodoc',    # auto-generate docs from docstrings
        'sphinx.ext.napoleon',   # parse NumPy and Google-style docstrings
        'sphinx.ext.doctest',    # run doctest examples in docs
        'sphinx.ext.viewcode',   # add links to source code
        'sphinx.ext.intersphinx',# link to NumPy, Python docs
    ]
    html_theme = 'sphinx_rtd_theme'

    # Napoleon settings
    napoleon_numpy_docstring = True
    napoleon_use_param = False
    napoleon_use_rtype = False

    # autodoc settings
    autodoc_typehints = 'description'
    autodoc_member_order = 'bysource'
""")

print('Sphinx conf.py:')
print(sphinx_conf)

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Docstring Quality and Sphinx Architecture', fontsize=14, fontweight='bold')

# --- Panel 1: Docstring section audit ---
ax1 = axes[0]
ax1.set_title('Docstring Section Audit', fontweight='bold')

functions = ['melting_temperature\n(complete)', 'bad_function\n(incomplete)']
sections_labels = ['Summary', 'Parameters', 'Returns', 'Raises', 'Examples', 'Notes', 'References']
presence = [
    [1, 1, 1, 1, 1, 1, 1],   # melting_temperature
    [1, 0, 0, 0, 0, 0, 0],   # bad_function
]

im = ax1.imshow(presence, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax1.set_xticks(range(len(sections_labels)))
ax1.set_xticklabels(sections_labels, rotation=30, ha='right', fontsize=9)
ax1.set_yticks(range(len(functions)))
ax1.set_yticklabels(functions, fontsize=9)

for i in range(len(functions)):
    for j in range(len(sections_labels)):
        text = 'YES' if presence[i][j] else 'NO'
        color = 'white'
        ax1.text(j, i, text, ha='center', va='center', fontsize=8,
                 color=color, fontweight='bold')

ax1.set_xlabel('Docstring Section')

# --- Panel 2: Sphinx build pipeline ---
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('Sphinx Build Pipeline', fontweight='bold')

pipeline_steps = [
    (5, 9.0, 'Source code\n+ docstrings', '#3498db'),
    (5, 7.0, 'sphinx-apidoc\n(generates .rst)', '#9b59b6'),
    (5, 5.0, 'sphinx-build\n(autodoc + napoleon)', '#e67e22'),
    (5, 3.0, 'HTML output\ndocs/_build/', '#27ae60'),
    (5, 1.0, 'Read the Docs\nor GitHub Pages', '#2c3e50'),
]

for x, y, label, color in pipeline_steps:
    bbox = mpatches.FancyBboxPatch((x - 2, y - 0.5), 4, 1,
                                    boxstyle='round,pad=0.1', facecolor=color,
                                    alpha=0.85, edgecolor='white', linewidth=2)
    ax2.add_patch(bbox)
    ax2.text(x, y, label, ha='center', va='center', fontsize=9,
             color='white', fontweight='bold')

# Draw arrows
for i in range(len(pipeline_steps) - 1):
    _, y1, _, _ = pipeline_steps[i]
    _, y2, _, _ = pipeline_steps[i + 1]
    ax2.annotate('', xy=(5, y2 + 0.5), xytext=(5, y1 - 0.5),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=2))

plt.tight_layout()
plt.savefig('documentation_overview.png', dpi=120, bbox_inches='tight')
plt.show()

## Exercises

See `exercises/README.md` → E03.

1. Write a fully documented `melting_temperature` function with all required NumPy-style sections.
2. Run `python -m doctest` on your file. Does every `>>>` example in the docstring produce the expected output?
3. What is the difference between a doctest and a pytest test? When would you use each?

## Quiz

1. What are the four required sections in a NumPy-style docstring?
2. What Sphinx extension parses NumPy-style docstrings?
3. What does `autodoc` do in Sphinx?
4. How do you run doctests from the command line?
5. What is the difference between a summary line and a Notes section in a docstring?

## Reflection

*After completing:* Why is the docstring written from the caller's perspective, not the implementer's? Give a specific example where the two perspectives would produce different text.

## References

- NumPy docstring guide: https://numpydoc.readthedocs.io/en/latest/format.html
- Sphinx autodoc: https://www.sphinx-doc.org/en/master/usage/extensions/autodoc.html